In [24]:
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption

from tqdm import tqdm
import re
import collections
import pymupdf

In [25]:
# Helper functions from pdf_parser.ipynb
def build_line_table(pdf_file_path, y_tolerance=2.0, line_tolerance=2.0):
    all_lines = []
    pagewise_lines = {}
    try:
        print("Opening PDF file...")
        doc = pymupdf.open(pdf_file_path)
        print("PDF file opened successfully.")
        print("Number of pages in PDF file:", doc.page_count)
    except Exception as e:
        print("Error occurred while opening PDF file:", e)
        return all_lines

    for page_num, page in tqdm(
        enumerate(doc), total=doc.page_count, desc="Processing pages"
    ):
        blocks = page.get_text("dict")["blocks"]
        for block_idx, block in enumerate(blocks):
            for line_idx, line in enumerate(block.get("lines", [])):
                spans = line.get("spans", [])
                if not spans:
                    continue

                # Sort spans by y (top to bottom) and then x (left to right) with tolerance
                # This is to handle cases where spans are not in reading order
                min_x = min(span["bbox"][0] for span in spans)
                min_y = min(span["bbox"][1] for span in spans)
                max_x = max(span["bbox"][2] for span in spans)
                max_y = max(span["bbox"][3] for span in spans)
                for span in spans:
                    x0, y0, _, _ = span["bbox"]
                    span["_y_round"] = round(y0 / y_tolerance) * y_tolerance
                    span["_x"] = x0
                spans_sorted = sorted(spans, key=lambda s: (s["_y_round"], s["_x"]))
                # spans = spans_sorted
                # Merge spans into one line string
                merged_text = " ".join(
                    span["text"].strip()
                    for span in spans_sorted
                    if span["text"].strip()
                )
                if not merged_text:
                    continue

                # Choose a "dominant" font/size for the line
                sizes = [span["size"] for span in spans_sorted]
                fonts = [span["font"] for span in spans_sorted]

                # Representative = most common size & font by length of text
                rep_size = max(set(sizes), key=lambda s: sum(len(span["text"]) for span in spans_sorted if span["size"] == s))
                rep_font = max(set(fonts), key=lambda f: sum(len(span["text"]) for span in spans_sorted if span["font"] == f))

                # Position = from first span
                x1, y1, x2, y2 = spans_sorted[0]["bbox"]
                x = x1
                y = (y1 + y2) / 2  # Midpoint of the first span vertically

                item = {
                    "page": page_num + 1,
                    "block": block_idx,
                    "line": line_idx,
                        "text": merged_text,
                        "size": rep_size,
                        "span_sizes": sizes,
                        "span_fonts": fonts,
                        "font": rep_font,
                        "bbox": (min_x, min_y, max_x, max_y),
                        "x": x,
                        "y": y,
                    }
                all_lines.append(item)
                if page_num + 1 not in pagewise_lines:
                    pagewise_lines[page_num + 1] = []
                pagewise_lines[page_num + 1].append(item)

    # Sort all lines by page, y (top to bottom), x (left to right)
    #

    for line in all_lines:
        line["_y_round"] = round(line["y"] / line_tolerance) * line_tolerance
    all_lines.sort(key=lambda l: (l["page"], l["_y_round"], l["x"]))

    return all_lines, pagewise_lines


def analyze_text(
    all_lines,
    min_heading_occ_count=5,
    min_heading_total_char_length=50,
    main_body_tolerance=0.5,
):


    size_occurrences = collections.Counter(line["size"] for line in all_lines)
    # Get lengths of text above main body size for headings total
    length_by_size = collections.Counter()
    length_by_font = collections.Counter()
    top_line_pos = []
    for line in all_lines:
        if line["line"] == 0:
            top_line_pos.append(line["y"])
        length_by_size[line["size"]] += len(line["text"])
        length_by_font[line["font"]] += len(line["text"])

    # Most common top line position
    most_common_top = collections.Counter(top_line_pos).most_common(1)[0][0]
    print("DEBUG: Most common top line position (y):", most_common_top)
    
    
    main_body_size = max(length_by_size, key=length_by_size.get)
    main_body_font = max(length_by_font, key=length_by_font.get)
    print("Determined main body font size:", main_body_size)

    headings = []
    for size, total_length in length_by_size.most_common():
        if (
            size > main_body_size + main_body_tolerance
            and total_length >= min_heading_total_char_length
            and size_occurrences[size] >= min_heading_occ_count
        ):
            print(
                f"  Possible heading size: {size} (total text length: {total_length} chars, occurrences: {size_occurrences[size]})"
            )
            headings.append((size, total_length, size_occurrences[size]))
    headings_sorted = [size for size, _, _ in sorted(headings, reverse=True)]

    # Get some more stats
    text_indent_by_size = collections.Counter()
    for line in all_lines:
        if line["size"] == main_body_size:
            text_indent_by_size[line["x"]] += 1

    main_body_indent = text_indent_by_size.most_common(1)[0][0]
   


    print("DEBUG: Text indent in main body:", main_body_indent)

    return main_body_size, main_body_font, main_body_indent, most_common_top, headings_sorted
def classify_lines(all_lines,main_body_size,heading_sizes,main_body_tolerance=0.5):
    for line in all_lines:
        line_size, txt, font = line["size"], line["text"], line["font"]
        label = None
        if line_size in heading_sizes:
            heading_level = heading_sizes.index(line_size) + 1
            label = f"heading_{heading_level}"
        elif abs(line_size - main_body_size) <= main_body_tolerance:
            label = "bodytext"
        elif line_size < main_body_size:
            label = "small" 
        else:
            label = "big"
        line["label"] = label

    return all_lines

def build_sections(all_lines, main_body_size, heading_sizes, main_body_tolerance=0.5):
    sections = []
    current = None
    for line in all_lines:
        sz, txt, font = line["size"], line["text"], line["font"]
        if sz in heading_sizes:
            if current:
                sections.append(current)
            current = {
                "id": len(sections) + 1,
                "heading": txt,
                "heading_level": heading_sizes.index(sz) + 1,
                "text": "",
                "font_size": sz,
                "font_style": font,
                "page": line["page"],
            }
        elif current and abs(sz - main_body_size) <= main_body_tolerance:
            if current["text"]:
                current["text"] += " "
            current["text"] += txt

    if current:
        sections.append(current)

    return sections


def clean_sections(sections):

    for i, section in enumerate(sections):
        if section is None:
            continue

        # merge consecutive empty same-level headings
        if (
            i < len(sections) - 1
            and section["heading_level"] == sections[i + 1]["heading_level"]
            and section["text"].strip() == ""
        ):
            while (
                i < len(sections) - 1
                and sections[i + 1]
                and section["heading_level"] == sections[i + 1]["heading_level"]
                and sections[i + 1]["text"].strip() == ""
            ):

                section["heading"] += " " + sections[i + 1]["heading"]
                sections[i + 1] = None
                i += 1

            # only copy text if the *next* section actually has body text
            if (
                i < len(sections) - 1
                and sections[i + 1]
                and sections[i + 1]["text"].strip() != ""
            ):
                section["text"] = sections[i + 1]["text"]
                section["heading"] += " " + sections[i + 1]["heading"]
                sections[i + 1] = None

    sections_filtered = [s for s in sections if s is not None]
    return sections_filtered

In [26]:
import pikepdf


In [27]:
pdf_file = "../grammar_books/mandan.pdf"

CLEAN_PDF = True
if CLEAN_PDF:
    with pikepdf.open(pdf_file,allow_overwriting_input=True) as pdf:
        pdf.save(pdf_file)

In [28]:


pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.accelerator_options = AcceleratorOptions(
        num_threads=4, device=AcceleratorDevice.AUTO
    )
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options.do_cell_matching = True

doc_converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )



doc = doc_converter.convert(pdf_file).document

2025-12-17 22:56:58,181 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]


2025-12-17 22:56:58,240 - INFO - Going to convert document batch...
2025-12-17 22:56:58,242 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 50a286a9d5fd8933113e2d1d283e9e7c
2025-12-17 22:56:58,242 - INFO - Accelerator device: 'cpu'
2025-12-17 22:56:59,048 - INFO - Accelerator device: 'cpu'
2025-12-17 22:56:59,705 - INFO - Processing document mandan.pdf
2025-12-17 23:09:51,612 - INFO - Finished converting document mandan.pdf in 773.45 sec.


In [29]:
table_bboxes = []
for table in doc.tables:
    for prov in table.prov:
        bbox = prov.bbox
        page_no = prov.page_no
        bbox_cleaned = (bbox.l, bbox.t, bbox.r, bbox.b)
        table_bboxes.append({"page_no": page_no, "bbox": bbox_cleaned})
print(len(table_bboxes), table_bboxes[0])


66 {'page_no': 5, 'bbox': (57.195556640625, 507.52391052246094, 413.828369140625, 74.38702392578125)}


In [30]:
# Load with pymupdf to classify sections
all_lines, pagewise_lines = build_line_table(pdf_file)

Opening PDF file...
PDF file opened successfully.
Number of pages in PDF file: 624


Processing pages: 100%|██████████| 624/624 [00:22<00:00, 27.86it/s] 


In [31]:
min_heading_occ_count = 5
min_heading_total_char_length = 50
main_body_tolerance = 0.4
max_heading_number = 4

print("DEBUG: Analyzing fonts to determine main body and headings...")
main_body_size, main_body_font, main_body_indent, most_common_top, headings_sorted = analyze_text(
    all_lines,
    min_heading_occ_count,
    min_heading_total_char_length,
    main_body_tolerance,
)

DEBUG: Analyzing fonts to determine main body and headings...
DEBUG: Most common top line position (y): 85.08844375610352
Determined main body font size: 10.909099578857422
  Possible heading size: 11.9552001953125 (total text length: 2095 chars, occurrences: 87)
  Possible heading size: 14.346199989318848 (total text length: 824 chars, occurrences: 36)
  Possible heading size: 20.662500381469727 (total text length: 213 chars, occurrences: 13)
DEBUG: Text indent in main body: 58.11000061035156


In [32]:
# Assume parallel sentence extraction is done and load from file
import json
language_name = pdf_file.split("/")[-1].replace(".pdf","")
print("DEBUG: Loading parallel sentences for language:", language_name)
with open(f"../data/parallel_sentences/{language_name}_parallel_sentences.json", "r", encoding="utf-8") as f:
    parallel_sentences = json.load(f)
    
print(len(parallel_sentences), "parallel sentences loaded.")

DEBUG: Loading parallel sentences for language: mandan
1421 parallel sentences loaded.


In [33]:
# Get block information for parallel sentences
parallel_blocks = {}
for sentence in parallel_sentences:
    page_num = sentence["page_num"]
    bboxes = sentence["bboxes"]
    source = sentence["source"]
    gloss = sentence["gloss"]
    translation = sentence["translation"]
    text_block = source + " | " + gloss + " | " + translation
    min_x0 = min(b[0] for b in bboxes)
    min_y0 = min(b[1] for b in bboxes)
    max_x1 = max(b[2] for b in bboxes)
    max_y1 = max(b[3] for b in bboxes)
    block_bbox = (min_x0, min_y0, max_x1, max_y1)
    
    if page_num not in parallel_blocks:
        parallel_blocks[page_num] = []
    parallel_blocks[page_num].append({
        "bbox": block_bbox,
        "text": text_block
    })



for page_num, blocks in parallel_blocks.items():
    print(f"Page {page_num} has {len(blocks)} parallel blocks.")
    print("Block bboxes:", blocks)

Page 53 has 3 parallel blocks.
Block bboxes: [{'bbox': (88.2959976196289, 201.8064422607422, 221.31565856933594, 257.4376220703125), 'text': 'a. [ˈikãm ĩ nĩ] | pv.ins-ins.frce-be.twisted | ‘to twist something up’'}, {'bbox': (88.12100219726562, 261.02325439453125, 185.94833374023438, 315.9406433105469), 'text': 'b. [mãˈnãːteʔʃ] | 1a-stand.up=ind.m | ‘I stood up’'}, {'bbox': (88.61199951171875, 318.8137613932292, 401.56646728515625, 374.4446105957031), 'text': 'c. [ˈmãːmãnãnũːnĩx ĩ nĩstoʔʃ] | neg-unsp-2a-steal=neg=2pl=pot=ind.m | ‘thou shalt not commit adultery’ (lit. ‘you should not steal anyone’)'}]
Page 56 has 1 parallel blocks.
Block bboxes: [{'bbox': (99.63400268554688, 516.7452392578125, 186.77096557617188, 571.6626586914062), 'text': 'a.  minihé’sh | 1a-2s-see=ind.m | ‘I see you.’'}]
Page 57 has 1 parallel blocks.
Block bboxes: [{'bbox': (88.12100219726562, 78.87025451660156, 175.44287109375, 133.7876434326172), 'text': 'b.  manahé’sh | 1s-2a-see=ind.m | ‘You see me.’'}]
Page 58 

In [34]:
def convert_docling_bbox_to_pymupdf(bbox, page_height):
    """
    Convert Docling bbox (bottom-left origin) into PyMuPDF bbox (top-left origin).
    bbox is an object with l, t, r, b attributes.
    """
    return (
        bbox.l,
        page_height - bbox.t,
        bbox.r,
        page_height - bbox.b,
    )


In [35]:
pdf_doc = pymupdf.open(pdf_file)
print(f"Loaded successfully doc with {pdf_doc.page_count} pages")

sections = []
current = None

parallel_added ={}

unique_labels = set()
for item,side in doc.iterate_items():

    bbox = item.prov[0].bbox
    page_no = item.prov[0].page_no
    label = item.label
    actual_label = label
    #print(f"Page no {page_no} Bounding Box: {bbox}")
    # print(f"{label} - Item: {item}")
    color = (1,1,1) # default white


    # Load pymupdf page
    page = pdf_doc.load_page(page_no-1)
    page_height = page.mediabox.height

    # pymupdf coordinate system has origin at top-left corner
    bbox_converted = convert_docling_bbox_to_pymupdf(bbox, page_height)
    rect = pymupdf.Rect(bbox_converted)
    header_height_threshold = page_height * 0.12 # 12% of page height

    x1, y1, x2, y2 = bbox_converted
    y_mid = (y1 + y2) / 2
    if y_mid < header_height_threshold:
        label = "page_header"    

    lines = pagewise_lines.get(page_no, [])

    if label=="section_header":
        section_header_lines = []
        for line in lines:
            if line["y"] >= y1 and line["y"] <= y2:
                section_header_lines.append(line["text"])
                #print(f"  Line within section header bbox: {line['text']}")
                font_size = line["size"]
                font_style = line["font"]
                if font_style == main_body_font and font_size == main_body_size:
                    #print("    WARNING: Section header line has same font and size as main body text.")
                    actual_label = "text"  # reclassify as body text
        if actual_label == "section_header":
            #print(f"Section header found on page {page_no} with text: {item.orig if hasattr(item, 'orig') else ''}")
            if font_size in headings_sorted:
                heading_level = headings_sorted.index(font_size) + 1
            elif font_size < min(headings_sorted):
                heading_level = len(headings_sorted) + 1
            else:
                heading_level = 0
            current = {
                "id": len(sections) + 1,
                "heading": " ".join(section_header_lines),
                "heading_level": heading_level,
                "text": "",
                "font_size": font_size,
                "font_style": font_style,
                "page": page_no,
                "paragraphs": [],
            }
            sections.append(current)
    if actual_label not in ["section_header","table","caption","picture","footnote","page_header"]:
        if current:
            # Get bbox
            text_bbox = item.prov[0].bbox
    
            text_bbox_pymupdf = convert_docling_bbox_to_pymupdf(text_bbox, page_height)
            parallel_sentence_blocks = parallel_blocks.get(page_no, [])
            eps = 0.2
            matched_idx = -1
            ps_text = ""
            for ps_idx,ps_block in enumerate(parallel_sentence_blocks):
                ps_bbox = ps_block["bbox"]
                
                # Check for overlap
                x0 = max(text_bbox_pymupdf[0], ps_bbox[0])
                y0 = max(text_bbox_pymupdf[1], ps_bbox[1])
                x1 = min(text_bbox_pymupdf[2], ps_bbox[2])
                y1 = min(text_bbox_pymupdf[3], ps_bbox[3])
                if x0 < x1 + eps and y0 < y1 + eps:
                    matched_idx = ps_idx
                    ps_text = ps_block["text"]
                    break

                
                # There is overlap
            if matched_idx != -1:
                
                # TODO: add parallel block into section?
                if page_no not in parallel_added:
                    parallel_added[page_no] = []
                    parallel_added[page_no].append(parallel_sentence_blocks[matched_idx])
                elif parallel_sentence_blocks[matched_idx] not in parallel_added[page_no]:
                    parallel_added[page_no].append(parallel_sentence_blocks[matched_idx])
                else:
                    continue
                # Add parallel block into section text
                print("Adding parallel sentence to section on page", page_no, " ps_text:", ps_text)
                current["text"] += ps_text + " "
                current["paragraphs"].append(ps_text)

                continue
            item_text = item.orig if hasattr(item, 'orig') else None
            if item_text:
                current["text"] += item_text
                current["paragraphs"].append(item_text)
        else:
            print(f"WARNING: Text item found outside of any section on page {page_no}: {item.orig if hasattr(item, 'orig') else ''}")
        

    unique_labels.add(actual_label)
    match actual_label:
        case "section_header":
            color = (1,0,0) # red
        case "text":
            color = (0.5,0.5,0.5) # grey
        case "table":
            color = (0,1,0) # green
        case "list_item":
            color = (0,0,1) # blue
        case "picture":
            color = (1,1,0) # yellow
        case "page_header":
            color = (1,0,1) # magenta
        case "footnote":
            color = (0,1,1) # cyan
        case "caption":
            color = (1,0.8,0) # orange
        case "code":
            color = (0.6,0.4,0.2) # brown
        case "formula":
            color = (0,0.1,0.9) # dark green
        case "document_index":
            color = (0.3,0.3,0.3) # dark grey
        case _:
            color = (0,0,0) # black

   
   

    #print(f"Drawing on page {page_no} bbox {bbox} with label {label} and color {color} - text: {item.orig if hasattr(item, 'orig') else ''}")
    page.draw_rect(rect, color=color, width=1)

if current:
    sections.append(current)

pdf_doc.save("annotated_output.pdf")
print("Unique labels in document items:")
for label in unique_labels:
    print(label)

Loaded successfully doc with 624 pages
Adding parallel sentence to section on page 53  ps_text: a. [ˈikãm ĩ nĩ] | pv.ins-ins.frce-be.twisted | ‘to twist something up’
Adding parallel sentence to section on page 53  ps_text: b. [mãˈnãːteʔʃ] | 1a-stand.up=ind.m | ‘I stood up’
Adding parallel sentence to section on page 53  ps_text: c. [ˈmãːmãnãnũːnĩx ĩ nĩstoʔʃ] | neg-unsp-2a-steal=neg=2pl=pot=ind.m | ‘thou shalt not commit adultery’ (lit. ‘you should not steal anyone’)
Adding parallel sentence to section on page 56  ps_text: a.  minihé’sh | 1a-2s-see=ind.m | ‘I see you.’
Adding parallel sentence to section on page 57  ps_text: b.  manahé’sh | 1s-2a-see=ind.m | ‘You see me.’
Adding parallel sentence to section on page 58  ps_text: minísweerut xí’hseena | horse#feces#eat be.old=def=dem.dist=top | ‘the old dog’ (Hollow 1973a: 189)
Adding parallel sentence to section on page 58  ps_text: Kóoxą’te Míihs tasúkseena | corn woman=def al-child=def=dem.dist=top | ‘The Corn Woman’s child’ (Hollow 1

In [36]:
for sec in sections:
    print(f"Section {sec['id']} (page {sec['page']}): {sec['heading']} Text length: {len(sec['text'])} chars")
    for para in sec['paragraphs']:
        print(f"  Paragraph ({len(para)} chars): {para}")

Section 1 (page 1): A grammar of Mandan Text length: 45 chars
  Paragraph (13 chars): Ryan M. Kasak
  Paragraph (32 chars): Comprehensive Grammar Library 10
Section 2 (page 2): Comprehensive Grammar Library Text length: 671 chars
  Paragraph (41 chars): Editor: Martin Haspelmath In this series:
  Paragraph (43 chars): 1. Jacques, Guillaume. A grammar of Japhug.
  Paragraph (37 chars): 2. Grimm, Nadine. A grammar of Gyeli.
  Paragraph (82 chars): 3. Maurer-Cecchini, Philippe. A grammar of Tuatschin: A Sursilvan Romansh dialect.
  Paragraph (40 chars): 4. Visser, Eline. A grammar of Kalamang.
  Paragraph (194 chars): 5. Caballero, Gabriela. A grammar of Choguita Rarámuri: In collaboration with Luz Elena León Ramírez, Sebastián Fuentes Holguín, Bertha Fuentes Loya and other Choguita Rarámuri language experts.
  Paragraph (57 chars): 6. Barlow, Russell. A grammar of Ulwa (Papua New Guinea).
  Paragraph (39 chars): 7. Terhart, Lena. A grammar of Paunaka.
  Paragraph (62 chars): 8. Pebley, C

In [37]:
print("Before cleaning, total sections:", len(sections))
cleaned_sections = clean_sections(sections)
print("After cleaning, total sections:", len(cleaned_sections))

Before cleaning, total sections: 277
After cleaning, total sections: 276


In [38]:
# Save sections
import json
from pathlib import Path
file_path = f"../data/sections/{Path(pdf_file).stem}_sections.json"
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_sections, f, ensure_ascii=False, indent=4)

print(f"Sections saved to {file_path}")

Sections saved to ../data/sections/mandan_sections.json
